In [ ]:
import argparse
import re
import sys
from datetime import datetime
import os

import pandas as pd
import requests

API_BASE = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

RAW_COLUMNS = [
    "state", "district", "market", "commodity", "variety",
    "grade", "arrival_date", "min_price", "max_price", "modal_price",
]


# 1.Ingestion

def fetch_live(api_key, states=None, commodities=None, limit=5000, pages=1):
    all_records = []
    for page in range(pages):
        params = {
            "api-key": api_key,
            "format": "json",
            "limit": limit,
            "offset": page * limit,
        }
        if states:
            params["filters[state]"] = states[0]
        if commodities:
            params["filters[commodity]"] = commodities[0]

        resp = requests.get(API_BASE, params=params, timeout=30)
        resp.raise_for_status()
        batch = resp.json().get("records", [])
        if not batch:
            break
        all_records.extend(batch)

    if not all_records:
        raise RuntimeError("No records returned -- check API key / filters.")

    return pd.DataFrame(all_records)[RAW_COLUMNS]


def fetch_demo(path="/content/demo_raw_sample.csv"):
    return pd.read_csv(path, dtype=str)


#2. cleaning/ transformation

def parse_date(value):
    if pd.isna(value) or str(value).strip() == "":
        return None
    value = str(value).strip()
    for fmt in ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(value, fmt).date().isoformat()
        except ValueError:
            continue
    return None


def standardise_name(value):
    if pd.isna(value):
        return None
    base = re.sub(r"\(.*?\)", "", str(value))
    base = re.sub(r"\s+", " ", base).strip().lower()
    return base.title()


def clean(df_raw):
    df = df_raw.copy()
    n_raw = len(df)

    for col in ["min_price", "max_price", "modal_price"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["arrival_date_iso"] = df["arrival_date"].apply(parse_date)
    n_bad_dates = df["arrival_date_iso"].isna().sum()
    df = df.dropna(subset=["arrival_date_iso"])

    df["commodity_std"] = df["commodity"].apply(standardise_name)
    df["variety_std"] = df["variety"].apply(standardise_name)

    dedup_keys = ["state", "district", "market", "commodity_std",
                  "variety_std", "arrival_date_iso"]
    n_before_dedup = len(df)
    df = df.drop_duplicates(subset=dedup_keys, keep="first")
    n_duplicates = n_before_dedup - len(df)

    anomaly_mask = df["max_price"] < df["min_price"]
    n_anomalies = int(anomaly_mask.sum())
    df["is_anomaly"] = anomaly_mask
    df.loc[anomaly_mask, ["min_price", "max_price", "modal_price"]] = None

    missing_modal_mask = df["modal_price"].isna() & ~anomaly_mask
    n_missing_modal = int(missing_modal_mask.sum())
    df["is_estimated"] = missing_modal_mask
    df.loc[missing_modal_mask, "modal_price"] = (
        (df.loc[missing_modal_mask, "min_price"] + df.loc[missing_modal_mask, "max_price"]) / 2
    )

    rejects = df[df["modal_price"].isna()]
    df = df[df["modal_price"].notna()]

    report = {
        "rows_raw": n_raw,
        "rows_bad_date_dropped": int(n_bad_dates),
        "rows_duplicate_removed": int(n_duplicates),
        "rows_flagged_anomaly": n_anomalies,
        "rows_modal_price_estimated": n_missing_modal,
        "rows_rejected_unusable": len(rejects),
        "rows_clean_output": len(df),
    }
    return df.reset_index(drop=True), report


# 3. CLI

def main():
    ap = argparse.ArgumentParser(description="MandiPulse ETL pipeline")
    ap.add_argument("--demo", action="store_true", help="use the synthetic offline fixture")
    ap.add_argument("--live", action="store_true", help="call the real data.gov.in API")
    ap.add_argument("--api-key", default=None)
    ap.add_argument("--states", nargs="*", default=None)
    ap.add_argument("--commodities", nargs="*", default=None)
    ap.add_argument("--limit", type=int, default=5000)
    ap.add_argument("--pages", type=int, default=1)
    ap.add_argument("--out", default="out/clean_prices.csv")
    args, unknown = ap.parse_known_args()

    if args.live:
        if not args.api_key:
            sys.exit("ERROR: --live requires --api-key (register free at data.gov.in)")
        raw = fetch_live(args.api_key, args.states, args.commodities, args.limit, args.pages)
        print(f"Pulled {len(raw)} raw rows from the LIVE data.gov.in API.")
    else:
        raw = fetch_demo()
        print(f"Loaded {len(raw)} rows from the SYNTHETIC demo fixture (not real data).")

    clean_df, report = clean(raw)

    print("\nCleaning report:")
    for k, v in report.items():
        print(f"  {k}: {v}")

    output_dir = os.path.dirname(args.out)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    clean_df.to_csv(args.out, index=False)
    print(f"\nClean output written to {args.out}")


if __name__ == "__main__":
    main()

Loaded 12 rows from the SYNTHETIC demo fixture (not real data).

Cleaning report:
  rows_raw: 12
  rows_bad_date_dropped: 1
  rows_duplicate_removed: 2
  rows_flagged_anomaly: 1
  rows_modal_price_estimated: 1
  rows_rejected_unusable: 1
  rows_clean_output: 8

Clean output written to out/clean_prices.csv
